In [1]:
import os
import urllib.parse
import requests
from astropy.io import ascii
from astropy.time import Time


IRSA_SEARCH = "https://irsa.ipac.caltech.edu/ibe/search/ztf/products/sci"
IRSA_DATA   = "https://irsa.ipac.caltech.edu/ibe/data/ztf/products/sci"


def _ensure_dir(path: str) -> None:
    os.makedirs(path, exist_ok=True)


def _to_jd(t_iso: str, scale: str = "utc") -> float:
    # Accepts "YYYY-MM-DD" or "YYYY-MM-DDTHH:MM:SS"
    return Time(t_iso, scale=scale).jd


def _query_metadata(where: str, timeout_s: int = 60):
    params = {"WHERE": where, "CT": "ipac_table"}  # CT is crucial
    full_url = f"{IRSA_SEARCH}?{urllib.parse.urlencode(params)}"

    r = requests.get(full_url, timeout=timeout_s)
    r.raise_for_status()

    text = r.text.strip()

    # When no results, IRSA may return a short message instead of an IPAC table
    if "No data returned" in text or len(text) < 50 or not text.startswith("\\"):
        return None, full_url, text

    table = ascii.read(text)
    if len(table) == 0:
        return None, full_url, text

    return table, full_url, None


def _sciimg_url_and_name(row) -> tuple[str, str]:
    filefracday = str(int(row["filefracday"]))  # robust conversion

    year  = filefracday[0:4]
    month = filefracday[4:6]
    day   = filefracday[6:8]
    frac  = filefracday[8:]

    fieldstr = f"{int(row['field']):06d}"
    ccdstr   = f"{int(row['ccdid']):02d}"
    filt     = str(row["filtercode"])
    q        = int(row["qid"])

    fname = (
        f"ztf_{filefracday}_{fieldstr}_{filt}"
        f"_c{ccdstr}_o_q{q}_sciimg.fits"
    )

    url = f"{IRSA_DATA}/{year}/{month}{day}/{frac}/{fname}"
    return url, fname


def _download_file(url: str, out_path: str, timeout_s: int = 120) -> bool:
    # True if downloaded; False if already present or failed
    if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
        return False

    with requests.get(url, stream=True, timeout=timeout_s) as r:
        if r.status_code != 200:
            print(f"    ! HTTP {r.status_code} for {url}")
            return False

        tmp_path = out_path + ".part"
        with open(tmp_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        os.replace(tmp_path, out_path)

    return True


def download_ztf_sciimg(
    *,
    fields,
    ccds,
    qids,
    filters,
    tmin,
    tmax,
    root_out="ztf_data",
    time_scale="utc",
    verbose=True,
):
    """
    Download ZTF sciimg.fits from IRSA for given selections.
    
    Parameters
    ----------
    fields, ccds, qids, filters : int/list[int] or str/list[str]
        You can pass a single value (e.g. fields=123) or a list (fields=[123,124]).
    tmin, tmax : str
        ISO date/time strings, e.g. "2023-01-01" or "2023-01-01T00:00:00".
    root_out : str
        Root directory where data folders are created.
    time_scale : str
        Usually "utc".
    """

    # normalize inputs to lists
    if not isinstance(fields, (list, tuple)):  fields  = [fields]
    if not isinstance(ccds, (list, tuple)):    ccds    = [ccds]
    if not isinstance(qids, (list, tuple)):    qids    = [qids]
    if not isinstance(filters, (list, tuple)): filters = [filters]
    
    _ensure_dir(root_out)

    tmin_jd = _to_jd(tmin, scale=time_scale)
    tmax_jd = _to_jd(tmax, scale=time_scale)

    if verbose:
        print(f"UTC range: {tmin} → {tmax}")
        print(f"JD range : {tmin_jd:.6f} → {tmax_jd:.6f}\n")

    for field in fields:
        for ccd in ccds:
            for qid in qids:
                for filt in filters:
                    folder = f"{field}_c{ccd}_q{qid}_{filt}\\sciimg"
                    out_dir = os.path.join(root_out, folder)
                    _ensure_dir(out_dir)

                    where = (
                        f"field={int(field)} AND ccdid={int(ccd)} AND qid={int(qid)} "
                        f"AND filtercode='{str(filt)}' "
                        f"AND obsjd>={tmin_jd} AND obsjd<={tmax_jd}"
                    )

                    table, full_url, err_text = _query_metadata(where)

                    if verbose:
                        print(f"[{folder}]")

                    if table is None:
                        if verbose:
                            print("  - No metadata returned.")
                            print(f"  - Query URL: {full_url}\n")
                        continue

                    downloaded = 0
                    skipped = 0

                    for row in table:
                        url, fname = _sciimg_url_and_name(row)
                        out_path = os.path.join(out_dir, fname)

                        if os.path.exists(out_path) and os.path.getsize(out_path) > 0:
                            skipped += 1
                            continue

                        if _download_file(url, out_path):
                            downloaded += 1

                    if verbose:
                        print(f"  - Images found     : {len(table)}")
                        print(f"  - Downloaded       : {downloaded}")
                        print(f"  - Already present  : {skipped}")
                        print(f"  - Output dir       : {out_dir}\n")


In [3]:
download_ztf_sciimg(
    fields=664,
    ccds=10,
    qids=4,
    filters="zr",
    tmin="2019-01-01T00:00:00",
    tmax="2023-05-01T23:59:59",
)


UTC range: 2019-01-01T00:00:00 → 2023-05-01T23:59:59
JD range : 2458484.500000 → 2460066.499988

[664_c10_q4_zr\sciimg]
  - Images found     : 631
  - Downloaded       : 0
  - Already present  : 631
  - Output dir       : ztf_data\664_c10_q4_zr\sciimg

